In [1]:
import pandas as pd
import os
import sys
import tomllib


sys.path.append(os.path.abspath("../.secrets"))
from db_config import get_connection

print ("Se conecto a la db")

Se conecto a la db


In [2]:
with open("../.secrets/secrets.toml", "rb") as f:
 config = tomllib.load(f)

ruta_archivo = config["paths"]["bronze_path"]

print ("Ruta del archivo: ", ruta_archivo)

Ruta del archivo:  assets/docs


In [3]:
conn = get_connection()
cursor = conn.cursor()

print ("Se conecto a la db y se obtuvo el cursor")

Se conecto a la db y se obtuvo el cursor


In [14]:
archivos = [f for f in os.listdir(f"../{ruta_archivo}") if f.endswith(".csv")]

print("archivos disponibles:")
for i, archivo in enumerate(archivos):
 print(f"{i} - {archivo}")

indice = int(input("Seleccione el índice del archivo a cargar: "))

if indice < 0 or indice >= len(archivos):
 print("Índice inválido.")
 exit()

archivo_seleccionado = archivos[indice]
print(f"Archivo seleccionado: {archivo_seleccionado}")
print(f'{archivo}')

archivos disponibles:
0 - clientes.csv
1 - productos.csv
2 - ventas.csv
Archivo seleccionado: clientes.csv
ventas.csv


In [16]:
from sqlalchemy import create_engine, inspect
import pandas as pd

# 1. Crear el Engine (esto es lo que SQLAlchemy e inspect necesitan)
# Extraemos los datos del config que ya cargaste en celdas anteriores
user = config["database"]["user"]
password = config["database"]["password"]
host = config["database"]["host"]
port = config["database"]["port"]
dbname = config["database"]["dbname"]

# Construimos la URL de conexión para PostgreSQL
engine = create_engine(f'postgresql://{user}:{password}@{host}:{port}/{dbname}')

# 2. Configuración de mapeo (la que ya teníamos)
mapeo_config = {
    "ventas.csv": "ventas_raw",
    "ventas_2024.csv": "ventas_raw",
    "productos.csv": "productos_raw",
    "clientes.csv": "clientes_raw"
}

if archivo_seleccionado in mapeo_config:
    tabla = mapeo_config[archivo_seleccionado]
    df = pd.read_csv(f"../{ruta_archivo}/{archivo_seleccionado}")

    # --- SECCIÓN DE TRADUCCIÓN ---
    if archivo_seleccionado == "clientes.csv":
        df = df.rename(columns={
            'ClienteID': 'id_cliente',
            'NombreCliente': 'nombre_cliente',
            'Ciudad': 'ciudad',
            'Segmento': 'segmento'
        })
    elif archivo_seleccionado == "productos.csv":
        # Asegúrate de que coincida con tu SQL: id_produto (vimos que faltaba una 'c')
        df = df.rename(columns={'ProductoID': 'id_produto', 'Nombre': 'nombre_producto'})
    # -----------------------------

    inspector = inspect(engine)
    columnas_db = [c['name'] for c in inspector.get_columns(tabla, schema='bronze')]
    
    if 'fuente_archivo' in columnas_db:
        df['fuente_archivo'] = archivo_seleccionado
    
    # Filtrar solo las que existen en la DB
    df_final = df[[c for c in df.columns if c in columnas_db]]
    
    print(f"Columnas detectadas para subir: {df_final.columns.tolist()}")

    try:
        df_final.to_sql(tabla, con=engine, schema='bronze', if_exists='replace', index=False, method='multi')
        print(f"✅ ¡Éxito! {len(df_final)} filas cargadas en bronze.{tabla}")
    except Exception as e:
        print(f"❌ Error: {e}")

Columnas detectadas para subir: ['id_cliente', 'nombre_cliente', 'genero', 'edad', 'ciudad', 'pais', 'segmento']
✅ ¡Éxito! 100 filas cargadas en bronze.clientes_raw
